In [0]:
WITH sent_emails AS (
  SELECT
    CASE journey_action_id
      WHEN 'bd448f28-0211-4e36-95f8-2e946c97fc26' THEN 'English - Active - Visit P1Y'
      WHEN '3cbba23f-a437-48f7-99dc-dee74a7c4fd6' THEN 'English - Active - No Visit P1Y'
      WHEN 'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94' THEN 'English - Cancelled - Visit P1Y'
      WHEN '991d60ac-7e3c-44a0-b7b1-b16afae1281d' THEN 'English - Cancelled - No Visit P1Y'
    END AS journey_label,
    Email_Address,
    message_profile_id
  FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback
  WHERE journey_action_id IN (
      'bd448f28-0211-4e36-95f8-2e946c97fc26',
      '3cbba23f-a437-48f7-99dc-dee74a7c4fd6',
      'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94',
      '991d60ac-7e3c-44a0-b7b1-b16afae1281d'
    )
    AND feedback_status = 'sent'
),

sent_cde_ids AS (
  SELECT
    s.journey_label,
    e.CDE_ID
  FROM sent_emails s
  INNER JOIN cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address e
    ON s.Email_Address = e.Email_Address
),

sent_profiles AS (
  SELECT DISTINCT
    sc.journey_label,
    sc.CDE_ID,
    sn.cc_user_profile_id,
    CASE
      WHEN sn.renewal_status LIKE 'Active%' THEN 'Active'
      WHEN sn.renewal_status LIKE 'Cancelled%' THEN 'Cancelled'
    END AS member_status
  FROM sent_cde_ids sc
  INNER JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
    ON sc.CDE_ID = sn.cde_id
  WHERE sn.snapshot_date_id = '20260513'
)

SELECT *
FROM sent_profiles
WHERE NOT (journey_label LIKE 'English - Active%' AND member_status = 'Cancelled')
  AND NOT (journey_label LIKE 'English - Cancelled%' AND member_status = 'Active')
;

-- Statement 2: Control Group (audience members with 5+ tickets who did NOT receive the email)
WITH treatment_cdes AS (
  SELECT DISTINCT e.CDE_ID
  FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback f
  INNER JOIN cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address e
    ON f.Email_Address = e.Email_Address
  WHERE f.journey_action_id IN (
    'bd448f28-0211-4e36-95f8-2e946c97fc26',
    '3cbba23f-a437-48f7-99dc-dee74a7c4fd6',
    'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94',
    '991d60ac-7e3c-44a0-b7b1-b16afae1281d'
  )
  AND f.feedback_status = 'sent'
),

eligible_members AS (
  SELECT
    sn.cde_id,
    sn.cc_user_profile_id,
    sn.recognitions_left,
    CASE
      WHEN sn.renewal_status LIKE 'Active%' THEN 'Active'
      WHEN sn.renewal_status LIKE 'Cancelled%' THEN 'Cancelled'
    END AS member_status
  FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
  WHERE sn.snapshot_date_id = '20260513'
    AND sn.recognitions_left >= 5
    AND (sn.renewal_status LIKE 'Active%' OR sn.renewal_status LIKE 'Cancelled%')
    AND sn.cde_id NOT IN (SELECT CDE_ID FROM treatment_cdes)
),

visitors_p12m AS (
  SELECT DISTINCT CDE_ID
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction
  WHERE VisitDateId BETWEEN 20250513 AND 20260513
    AND TicketFLAG = 'Yes'
)

SELECT
  CASE
    WHEN em.member_status = 'Active'    AND v.CDE_ID IS NOT NULL THEN 'English - Active - Visit P1Y'
    WHEN em.member_status = 'Active'    AND v.CDE_ID IS NULL     THEN 'English - Active - No Visit P1Y'
    WHEN em.member_status = 'Cancelled' AND v.CDE_ID IS NOT NULL THEN 'English - Cancelled - Visit P1Y'
    WHEN em.member_status = 'Cancelled' AND v.CDE_ID IS NULL     THEN 'English - Cancelled - No Visit P1Y'
  END AS journey_label,
  em.cde_id,
  em.cc_user_profile_id,
  em.member_status
FROM eligible_members em
LEFT JOIN visitors_p12m v ON em.cde_id = v.CDE_ID
;

-- Statement 3: Ticket Usage KPI — CineClub redemptions (May 13 – June 13), Treatment vs Control by segment
WITH sent_emails AS (
  SELECT
    CASE journey_action_id
      WHEN 'bd448f28-0211-4e36-95f8-2e946c97fc26' THEN 'English - Active - Visit P1Y'
      WHEN '3cbba23f-a437-48f7-99dc-dee74a7c4fd6' THEN 'English - Active - No Visit P1Y'
      WHEN 'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94' THEN 'English - Cancelled - Visit P1Y'
      WHEN '991d60ac-7e3c-44a0-b7b1-b16afae1281d' THEN 'English - Cancelled - No Visit P1Y'
    END AS journey_label,
    Email_Address
  FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback
  WHERE journey_action_id IN (
    'bd448f28-0211-4e36-95f8-2e946c97fc26',
    '3cbba23f-a437-48f7-99dc-dee74a7c4fd6',
    'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94',
    '991d60ac-7e3c-44a0-b7b1-b16afae1281d'
  )
  AND feedback_status = 'sent'
),

treatment_cdes AS (
  SELECT DISTINCT s.journey_label, e.CDE_ID
  FROM sent_emails s
  INNER JOIN cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address e
    ON s.Email_Address = e.Email_Address
),

treatment_profiles AS (
  SELECT DISTINCT
    tc.journey_label,
    tc.CDE_ID,
    'Treatment' AS group_type
  FROM treatment_cdes tc
  INNER JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
    ON tc.CDE_ID = sn.cde_id
  WHERE sn.snapshot_date_id = '20260513'
    AND NOT (tc.journey_label LIKE 'English - Active%' AND sn.renewal_status LIKE 'Cancelled%')
    AND NOT (tc.journey_label LIKE 'English - Cancelled%' AND sn.renewal_status LIKE 'Active%')
),

control_eligible AS (
  SELECT
    sn.cde_id AS CDE_ID,
    CASE
      WHEN sn.renewal_status LIKE 'Active%' THEN 'Active'
      WHEN sn.renewal_status LIKE 'Cancelled%' THEN 'Cancelled'
    END AS member_status
  FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
  WHERE sn.snapshot_date_id = '20260513'
    AND sn.recognitions_left >= 5
    AND (sn.renewal_status LIKE 'Active%' OR sn.renewal_status LIKE 'Cancelled%')
    AND sn.cde_id NOT IN (SELECT CDE_ID FROM treatment_cdes)
),

visitors_p12m AS (
  SELECT DISTINCT CDE_ID
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction
  WHERE VisitDateId BETWEEN 20250513 AND 20260513
    AND TicketFLAG = 'Yes'
),

control_profiles AS (
  SELECT
    CASE
      WHEN ce.member_status = 'Active'    AND v.CDE_ID IS NOT NULL THEN 'English - Active - Visit P1Y'
      WHEN ce.member_status = 'Active'    AND v.CDE_ID IS NULL     THEN 'English - Active - No Visit P1Y'
      WHEN ce.member_status = 'Cancelled' AND v.CDE_ID IS NOT NULL THEN 'English - Cancelled - Visit P1Y'
      WHEN ce.member_status = 'Cancelled' AND v.CDE_ID IS NULL     THEN 'English - Cancelled - No Visit P1Y'
    END AS journey_label,
    ce.CDE_ID,
    'Control' AS group_type
  FROM control_eligible ce
  LEFT JOIN visitors_p12m v ON ce.CDE_ID = v.CDE_ID
),

all_members AS (
  SELECT journey_label, CDE_ID, group_type FROM treatment_profiles
  UNION ALL
  SELECT journey_label, CDE_ID, group_type FROM control_profiles
),

-- Pre-aggregate redemptions per CDE_ID (forces view materialization before LEFT JOIN)
cineclub_redemptions AS (
  SELECT CDE_ID, COUNT(*) AS redemption_count
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction
  WHERE CineClub_Discount_Type_ID IN (5, 6)
    AND VisitDateId BETWEEN 20260513 AND 20260613
  GROUP BY CDE_ID
),

member_redemptions AS (
  SELECT
    am.group_type,
    am.journey_label,
    am.CDE_ID,
    COALESCE(r.redemption_count, 0) AS redemption_count
  FROM all_members am
  LEFT JOIN cineclub_redemptions r ON am.CDE_ID = r.CDE_ID
),

segment_summary AS (
  SELECT
    journey_label,
    group_type,
    COUNT(*) AS total_members,
    SUM(CASE WHEN redemption_count > 0 THEN 1 ELSE 0 END) AS members_redeemed,
    SUM(redemption_count) AS total_redemptions,
    ROUND(SUM(CASE WHEN redemption_count > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS redemption_rate_pct,
    ROUND(SUM(redemption_count) * 1.0 / COUNT(*), 2) AS avg_redemptions_per_member
  FROM member_redemptions
  GROUP BY journey_label, group_type
)

SELECT
  t.journey_label,
  t.group_type,
  t.total_members,
  t.members_redeemed,
  t.total_redemptions,
  t.redemption_rate_pct,
  t.avg_redemptions_per_member,
  ROUND((t.redemption_rate_pct - c.redemption_rate_pct) / NULLIF(c.redemption_rate_pct, 0) * 100, 2) AS rate_change_pct,
  ROUND((t.avg_redemptions_per_member - c.avg_redemptions_per_member) / NULLIF(c.avg_redemptions_per_member, 0) * 100, 2) AS avg_redeem_change_pct
FROM segment_summary t
LEFT JOIN segment_summary c
  ON t.journey_label = c.journey_label
  AND c.group_type = 'Control'
ORDER BY t.journey_label, t.group_type
;

-- Statement 4: Churn KPI — Active members on May 13 who are now Cancelled, Treatment vs Control
WITH sent_emails AS (
  SELECT
    CASE journey_action_id
      WHEN 'bd448f28-0211-4e36-95f8-2e946c97fc26' THEN 'English - Active - Visit P1Y'
      WHEN '3cbba23f-a437-48f7-99dc-dee74a7c4fd6' THEN 'English - Active - No Visit P1Y'
    END AS journey_label,
    Email_Address
  FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback
  WHERE journey_action_id IN (
    'bd448f28-0211-4e36-95f8-2e946c97fc26',
    '3cbba23f-a437-48f7-99dc-dee74a7c4fd6'
  )
  AND feedback_status = 'sent'
),

treatment_cdes AS (
  SELECT DISTINCT s.journey_label, e.CDE_ID
  FROM sent_emails s
  INNER JOIN cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address e
    ON s.Email_Address = e.Email_Address
),

-- Treatment: Active members who received the email
treatment_active AS (
  SELECT DISTINCT
    tc.journey_label,
    tc.CDE_ID,
    'Treatment' AS group_type
  FROM treatment_cdes tc
  INNER JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
    ON tc.CDE_ID = sn.cde_id
  WHERE sn.snapshot_date_id = '20260513'
    AND sn.renewal_status LIKE 'Active%'
),

-- All treatment CDE_IDs (all 4 journeys) for exclusion from control
all_treatment_cdes AS (
  SELECT DISTINCT e.CDE_ID
  FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback f
  INNER JOIN cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address e
    ON f.Email_Address = e.Email_Address
  WHERE f.journey_action_id IN (
    'bd448f28-0211-4e36-95f8-2e946c97fc26',
    '3cbba23f-a437-48f7-99dc-dee74a7c4fd6',
    'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94',
    '991d60ac-7e3c-44a0-b7b1-b16afae1281d'
  )
  AND f.feedback_status = 'sent'
),

-- Control: Active members with 5+ tickets who did NOT receive any email
control_active AS (
  SELECT sn.cde_id AS CDE_ID
  FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
  WHERE sn.snapshot_date_id = '20260513'
    AND sn.recognitions_left >= 5
    AND sn.renewal_status LIKE 'Active%'
    AND sn.cde_id NOT IN (SELECT CDE_ID FROM all_treatment_cdes)
),

visitors_p12m AS (
  SELECT DISTINCT CDE_ID
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction
  WHERE VisitDateId BETWEEN 20250513 AND 20260513
    AND TicketFLAG = 'Yes'
),

control_labeled AS (
  SELECT
    CASE
      WHEN v.CDE_ID IS NOT NULL THEN 'English - Active - Visit P1Y'
      ELSE 'English - Active - No Visit P1Y'
    END AS journey_label,
    ca.CDE_ID,
    'Control' AS group_type
  FROM control_active ca
  LEFT JOIN visitors_p12m v ON ca.CDE_ID = v.CDE_ID
),

all_active AS (
  SELECT journey_label, CDE_ID, group_type FROM treatment_active
  UNION ALL
  SELECT journey_label, CDE_ID, group_type FROM control_labeled
),

-- Status at 30-day mark (June 13)
status_jun13 AS (
  SELECT cde_id,
    CASE
      WHEN renewal_status LIKE 'Active%' THEN 'Active'
      WHEN renewal_status LIKE 'Cancelled%' THEN 'Cancelled'
      ELSE renewal_status
    END AS current_status
  FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
  WHERE snapshot_date_id = '20260613'
),

member_churn AS (
  SELECT
    a.group_type,
    a.journey_label,
    a.CDE_ID,
    COALESCE(cs.current_status, 'Not Found') AS current_status,
    CASE WHEN COALESCE(cs.current_status, 'Not Found') != 'Active' THEN 1 ELSE 0 END AS churned
  FROM all_active a
  LEFT JOIN status_jun13 cs ON a.CDE_ID = cs.cde_id
),

segment_summary AS (
  SELECT
    journey_label,
    group_type,
    COUNT(*) AS total_active_may13,
    SUM(churned) AS members_churned,
    ROUND(SUM(churned) * 100.0 / COUNT(*), 2) AS churn_rate_pct
  FROM member_churn
  GROUP BY journey_label, group_type
)

SELECT
  t.journey_label,
  t.group_type,
  t.total_active_may13,
  t.members_churned,
  t.churn_rate_pct,
  ROUND((t.churn_rate_pct - c.churn_rate_pct) / NULLIF(c.churn_rate_pct, 0) * 100, 2) AS churn_change_pct
FROM segment_summary t
LEFT JOIN segment_summary c
  ON t.journey_label = c.journey_label
  AND c.group_type = 'Control'
ORDER BY t.journey_label, t.group_type
;

-- Statement 5: Return Visits KPI — "No Visit P12M" members who visited within 30 days, Treatment vs Control
WITH sent_emails AS (
  SELECT
    CASE journey_action_id
      WHEN '3cbba23f-a437-48f7-99dc-dee74a7c4fd6' THEN 'English - Active - No Visit P1Y'
      WHEN '991d60ac-7e3c-44a0-b7b1-b16afae1281d' THEN 'English - Cancelled - No Visit P1Y'
    END AS journey_label,
    Email_Address
  FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback
  WHERE journey_action_id IN (
    '3cbba23f-a437-48f7-99dc-dee74a7c4fd6',
    '991d60ac-7e3c-44a0-b7b1-b16afae1281d'
  )
  AND feedback_status = 'sent'
),

treatment_cdes AS (
  SELECT DISTINCT s.journey_label, e.CDE_ID
  FROM sent_emails s
  INNER JOIN cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address e
    ON s.Email_Address = e.Email_Address
),

treatment_no_visit AS (
  SELECT DISTINCT
    tc.journey_label,
    tc.CDE_ID,
    'Treatment' AS group_type
  FROM treatment_cdes tc
  INNER JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
    ON tc.CDE_ID = sn.cde_id
  WHERE sn.snapshot_date_id = '20260513'
    AND NOT (tc.journey_label LIKE 'English - Active%' AND sn.renewal_status LIKE 'Cancelled%')
    AND NOT (tc.journey_label LIKE 'English - Cancelled%' AND sn.renewal_status LIKE 'Active%')
),

all_treatment_cdes AS (
  SELECT DISTINCT e.CDE_ID
  FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback f
  INNER JOIN cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address e
    ON f.Email_Address = e.Email_Address
  WHERE f.journey_action_id IN (
    'bd448f28-0211-4e36-95f8-2e946c97fc26',
    '3cbba23f-a437-48f7-99dc-dee74a7c4fd6',
    'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94',
    '991d60ac-7e3c-44a0-b7b1-b16afae1281d'
  )
  AND f.feedback_status = 'sent'
),

control_eligible AS (
  SELECT
    sn.cde_id AS CDE_ID,
    CASE
      WHEN sn.renewal_status LIKE 'Active%' THEN 'Active'
      WHEN sn.renewal_status LIKE 'Cancelled%' THEN 'Cancelled'
    END AS member_status
  FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
  WHERE sn.snapshot_date_id = '20260513'
    AND sn.recognitions_left >= 5
    AND (sn.renewal_status LIKE 'Active%' OR sn.renewal_status LIKE 'Cancelled%')
    AND sn.cde_id NOT IN (SELECT CDE_ID FROM all_treatment_cdes)
),

visitors_p12m AS (
  SELECT DISTINCT CDE_ID
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction
  WHERE VisitDateId BETWEEN 20250513 AND 20260513
    AND TicketFLAG = 'Yes'
),

control_no_visit AS (
  SELECT
    CASE
      WHEN ce.member_status = 'Active' THEN 'English - Active - No Visit P1Y'
      WHEN ce.member_status = 'Cancelled' THEN 'English - Cancelled - No Visit P1Y'
    END AS journey_label,
    ce.CDE_ID,
    'Control' AS group_type
  FROM control_eligible ce
  LEFT JOIN visitors_p12m v ON ce.CDE_ID = v.CDE_ID
  WHERE v.CDE_ID IS NULL
),

all_no_visit AS (
  SELECT journey_label, CDE_ID, group_type FROM treatment_no_visit
  UNION ALL
  SELECT journey_label, CDE_ID, group_type FROM control_no_visit
),

-- Pre-aggregate visits in 30-day window (forces view materialization)
visits_in_window AS (
  SELECT DISTINCT CDE_ID
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction
  WHERE VisitDateId BETWEEN 20260513 AND 20260613
    AND TicketFLAG = 'Yes'
),

member_visits AS (
  SELECT
    a.group_type,
    a.journey_label,
    a.CDE_ID,
    CASE WHEN vw.CDE_ID IS NOT NULL THEN 1 ELSE 0 END AS visited
  FROM all_no_visit a
  LEFT JOIN visits_in_window vw ON a.CDE_ID = vw.CDE_ID
),

segment_summary AS (
  SELECT
    journey_label,
    group_type,
    COUNT(*) AS total_no_visit_members,
    SUM(visited) AS members_returned,
    ROUND(SUM(visited) * 100.0 / COUNT(*), 2) AS return_visit_rate_pct
  FROM member_visits
  GROUP BY journey_label, group_type
)

SELECT
  t.journey_label,
  t.group_type,
  t.total_no_visit_members,
  t.members_returned,
  t.return_visit_rate_pct,
  ROUND((t.return_visit_rate_pct - c.return_visit_rate_pct) / NULLIF(c.return_visit_rate_pct, 0) * 100, 2) AS visit_change_pct
FROM segment_summary t
LEFT JOIN segment_summary c
  ON t.journey_label = c.journey_label
  AND c.group_type = 'Control'
ORDER BY t.journey_label, t.group_type
;

-- Statement 6: Re-signups KPI — Cancelled members on May 13 who are now Active, Treatment vs Control
WITH sent_emails AS (
  SELECT
    CASE journey_action_id
      WHEN 'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94' THEN 'English - Cancelled - Visit P1Y'
      WHEN '991d60ac-7e3c-44a0-b7b1-b16afae1281d' THEN 'English - Cancelled - No Visit P1Y'
    END AS journey_label,
    Email_Address
  FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback
  WHERE journey_action_id IN (
    'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94',
    '991d60ac-7e3c-44a0-b7b1-b16afae1281d'
  )
  AND feedback_status = 'sent'
),

treatment_cdes AS (
  SELECT DISTINCT s.journey_label, e.CDE_ID
  FROM sent_emails s
  INNER JOIN cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address e
    ON s.Email_Address = e.Email_Address
),

treatment_cancelled AS (
  SELECT DISTINCT
    tc.journey_label,
    tc.CDE_ID,
    'Treatment' AS group_type
  FROM treatment_cdes tc
  INNER JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
    ON tc.CDE_ID = sn.cde_id
  WHERE sn.snapshot_date_id = '20260513'
    AND sn.renewal_status LIKE 'Cancelled%'
),

all_treatment_cdes AS (
  SELECT DISTINCT e.CDE_ID
  FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback f
  INNER JOIN cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address e
    ON f.Email_Address = e.Email_Address
  WHERE f.journey_action_id IN (
    'bd448f28-0211-4e36-95f8-2e946c97fc26',
    '3cbba23f-a437-48f7-99dc-dee74a7c4fd6',
    'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94',
    '991d60ac-7e3c-44a0-b7b1-b16afae1281d'
  )
  AND f.feedback_status = 'sent'
),

control_cancelled AS (
  SELECT sn.cde_id AS CDE_ID
  FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
  WHERE sn.snapshot_date_id = '20260513'
    AND sn.recognitions_left >= 5
    AND sn.renewal_status LIKE 'Cancelled%'
    AND sn.cde_id NOT IN (SELECT CDE_ID FROM all_treatment_cdes)
),

visitors_p12m AS (
  SELECT DISTINCT CDE_ID
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction
  WHERE VisitDateId BETWEEN 20250513 AND 20260513
    AND TicketFLAG = 'Yes'
),

control_labeled AS (
  SELECT
    CASE
      WHEN v.CDE_ID IS NOT NULL THEN 'English - Cancelled - Visit P1Y'
      ELSE 'English - Cancelled - No Visit P1Y'
    END AS journey_label,
    cc.CDE_ID,
    'Control' AS group_type
  FROM control_cancelled cc
  LEFT JOIN visitors_p12m v ON cc.CDE_ID = v.CDE_ID
),

all_cancelled AS (
  SELECT journey_label, CDE_ID, group_type FROM treatment_cancelled
  UNION ALL
  SELECT journey_label, CDE_ID, group_type FROM control_labeled
),

-- Status at 30-day mark (June 13)
status_jun13 AS (
  SELECT cde_id,
    CASE
      WHEN renewal_status LIKE 'Active%' THEN 'Active'
      ELSE 'Not Active'
    END AS current_status
  FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
  WHERE snapshot_date_id = '20260613'
),

member_resignup AS (
  SELECT
    a.group_type,
    a.journey_label,
    a.CDE_ID,
    CASE WHEN COALESCE(cs.current_status, 'Not Active') = 'Active' THEN 1 ELSE 0 END AS re_signed_up
  FROM all_cancelled a
  LEFT JOIN status_jun13 cs ON a.CDE_ID = cs.cde_id
),

segment_summary AS (
  SELECT
    journey_label,
    group_type,
    COUNT(*) AS total_cancelled_may13,
    SUM(re_signed_up) AS members_re_signed_up,
    ROUND(SUM(re_signed_up) * 100.0 / COUNT(*), 2) AS resignup_rate_pct
  FROM member_resignup
  GROUP BY journey_label, group_type
)

SELECT
  t.journey_label,
  t.group_type,
  t.total_cancelled_may13,
  t.members_re_signed_up,
  t.resignup_rate_pct,
  ROUND((t.resignup_rate_pct - c.resignup_rate_pct) / NULLIF(c.resignup_rate_pct, 0) * 100, 2) AS resignup_change_pct
FROM segment_summary t
LEFT JOIN segment_summary c
  ON t.journey_label = c.journey_label
  AND c.group_type = 'Control'
ORDER BY t.journey_label, t.group_type
